#  A Deep Research agent 

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool , set_tracing_disabled ,  OpenAIChatCompletionsModel
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from openai import AsyncOpenAI
import asyncio
import os
from typing import Dict
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)

In [ ]:
set_tracing_disabled(True)

openai_client = AsyncOpenAI(base_url=os.getenv("GITHUB_BASE_URL"), api_key=os.getenv("GITHUB_TOKEN"))

model = OpenAIChatCompletionsModel(model="gpt-4o-mini", openai_client=openai_client)

### OpenAI Hosted Tools 

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."


# hosted tools available only in openai not through github marketplace 

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",  # using openai api key 
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message = "latest AI frameworks in 2026"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

### We will now use structured outputs 

In [ ]:


HOW_MANY_SEARCHES = 3  # now of searches we want to perform 

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"

class WebSearchItem(BaseModel):
    # adding reason also so that llm will choose the best search term to use for the query with the reasoning behind it
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=model,  # i use here model from github marketplace as openai giving rate limit for this agent 
    output_type=WebSearchPlan,
)

In [ ]:

message = "Latest AI Agent frameworks in 2026"

# with trace("Search"):  # use trace for tracing on openai platform
result = await Runner.run(planner_agent, message)
print(result.final_output)

In [ ]:
# send email tool 
@function_tool
def send_html_email(subject:str,html_body: str) -> Dict[str, str]:
    try: 
        import resend

        resend.api_key = os.getenv("RESEND_API_KEY")

        params: resend.Emails.SendParams = {
        "from": "onboarding@resend.dev",
        "to": os.getenv("TO_EMAIL"),
        "subject": subject,
        "html": html_body,
        }

        email = resend.Emails.send(params)
        print(email)
        return {"status": "success"}
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise

In [ ]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""


# email agent to send email uses send_html_email tool 
email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_html_email],
    model=model,
)


In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=model ,
    output_type=ReportData,
)

In [ ]:
# creating function to plan and execute the searches using planner_agent and search_agent together

async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

In [ ]:
# function to generate report using writer_agent and email_agent to send the report via email
async def generate_report(query: str, search_results: list[str]):
    """ Use the writer_agent to generate a report based on the search results """
    print("Generating report...")
    input = f"Query: {query}\n Search Results:\n" + "\n".join(search_results)
    result = await Runner.run(writer_agent, input)
    print("Finished generating report")
    return result.final_output


async def send_email(reportData : ReportData):
    '''Sending email of report '''
    print("writing email...")
    report = await Runner.run(email_agent,reportData.markdown_report)
    print("email send")
    return report
    

In [ ]:
# now research using all the agent 
query ="Latest AI Agent frameworks in 2025"

print("starting search... ")
search_plan =await plan_searches(query)
search_results = await perform_searches(search_plan)
report = await generate_report(query, search_results)
await send_email(report)  
print("done search!")